# Chronicle Runtime — Mistral-7B Benchmark (A10 GPU)

Run this notebook top-to-bottom on a **Colab Pro A10 (or A100) GPU runtime**.
It will produce the throughput gain and load-test numbers for the resume line:

> **Chronicle | Python, PyTorch, CUDA, FastAPI**  
> Built LLM inference runtime with timeout-based micro-batching and KV-cache reuse  
> Benchmarked on Mistral-7B (fp16, A10 GPU): **X% throughput gain** over HuggingFace baseline at batch_size=8  
> Load tested: **<Yms p95** at **Z concurrent requests** with async httpx harness

All numbers come from reproducible scripts — replace X/Y/Z with your actual output.

## Step 1 — Verify GPU

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip())
assert result.returncode == 0, 'No GPU found — switch runtime to GPU'

import torch
assert torch.cuda.is_available(), 'torch cannot see CUDA'
print(f'torch {torch.__version__}, CUDA {torch.version.cuda}')

## Step 2 — Install Chronicle

In [ ]:
import os
# If running from a cloned repo, install in editable mode.
# If running from Drive/upload, adjust the path below.
REPO_ROOT = os.path.abspath('..')  # notebooks/ is one level below repo root
if not os.path.exists(os.path.join(REPO_ROOT, 'pyproject.toml')):
    # Fallback: clone from GitHub (replace with your actual repo URL)
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_USER/chronicle', '/content/chronicle'],
                   check=True)
    REPO_ROOT = '/content/chronicle'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_ROOT, '-q'], check=True)
print('Installed.')

## Step 3 — Configure environment

In [ ]:
import os

os.environ['MODEL_NAME']       = 'mistralai/Mistral-7B-v0.1'
os.environ['DEVICE']           = 'cuda'
os.environ['MAX_BATCH']        = '8'
os.environ['BATCH_WINDOW_MS']  = '50'
os.environ['MAX_QUEUE_WAIT_MS']= '5000'
os.environ['BENCH_PROMPTS']    = '50'
os.environ['BENCH_TOKENS']     = '128'
os.environ['BENCH_BATCH_SIZE'] = '8'
os.environ['BENCH_WARMUP']     = '3'

# Optional: set HF token if Mistral-7B requires gated access
# os.environ['HUGGING_FACE_HUB_TOKEN'] = 'hf_...'

print('Environment configured.')

## Step 4 — Start Chronicle server

In [ ]:
import subprocess, time, httpx, os

env = {**os.environ}
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'chronicle_runtime.server.main:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Poll until healthy (model load can take 60-120s for Mistral-7B)
print('Waiting for server to load Mistral-7B...', end='')
for _ in range(180):
    time.sleep(1)
    try:
        r = httpx.get('http://localhost:8000/healthz', timeout=2)
        if r.status_code == 200:
            print(' ready!')
            break
    except Exception:
        pass
    print('.', end='', flush=True)
else:
    # Print server output for debugging
    out, _ = server_proc.communicate(timeout=5)
    print('\nServer failed to start. Output:')
    print(out.decode())
    raise RuntimeError('Server did not become healthy within 180s')

## Step 5 — Baseline benchmark (HuggingFace sequential generate)

In [ ]:
import subprocess, json, os, sys

os.makedirs('.bench', exist_ok=True)

result = subprocess.run(
    [sys.executable, '-m', 'chronicle_runtime.bench.run_baseline',
     '-o', '.bench/baseline.json'],
    env=os.environ,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Baseline benchmark failed')

with open('.bench/baseline.json') as f:
    baseline = json.load(f)

print(f"Baseline — {baseline['model_name']} on {baseline['device']}")
print(f"  {baseline['tokens_per_sec']:.1f} tokens/sec")
print(f"  {baseline['num_prompts']} prompts × {baseline['max_new_tokens']} max_new_tokens")
print(f"  gpu_mem_mb: {baseline.get('gpu_mem_mb', 'N/A')}")

## Step 6 — Chronicle benchmark (batched prefill + KV-cache reuse, batch_size=8)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'chronicle_runtime.bench.run_chronicle',
     '-o', '.bench/chronicle.json'],
    env=os.environ,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Chronicle benchmark failed')

with open('.bench/chronicle.json') as f:
    chronicle = json.load(f)

gain_pct = (chronicle['tokens_per_sec'] - baseline['tokens_per_sec']) / baseline['tokens_per_sec'] * 100

print(f"Chronicle — {chronicle['model_name']} on {chronicle['device']}")
print(f"  {chronicle['tokens_per_sec']:.1f} tokens/sec")
print(f"  batch_sizes used: {chronicle['batch_sizes']}")
print(f"  gpu_mem_mb: {chronicle.get('gpu_mem_mb', 'N/A')}")
print()
print(f">>> Throughput gain: {gain_pct:.1f}% over baseline <<<")

## Step 7 — Load test sweep (10 → 250 concurrent requests)

In [ ]:
# --sweep-requests 5 means each concurrency level sends level×5 total requests.
# Increase to 10 for tighter percentiles; decrease if time is limited.
result = subprocess.run(
    [sys.executable, '-m', 'chronicle_runtime.load.run_load',
     '--sweep', '--sweep-requests', '5',
     '--max-new-tokens', '128'],
    env=os.environ,
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 8 — Resume-ready summary

In [ ]:
import json

with open('.bench/chronicle.json') as f:
    chronicle = json.load(f)
with open('.bench/baseline.json') as f:
    baseline = json.load(f)
with open('.bench/load_sweep.json') as f:
    sweep = json.load(f)

gain_pct = (chronicle['tokens_per_sec'] - baseline['tokens_per_sec']) / baseline['tokens_per_sec'] * 100

# Find the highest concurrency level where p95 < 300ms and error_count == 0
best_level = None
for row in sorted(sweep['levels'], key=lambda r: r['concurrency'], reverse=True):
    if row['error_count'] == 0 and row['p95_ms'] < 300:
        best_level = row
        break
if best_level is None:
    best_level = min(sweep['levels'], key=lambda r: r['p95_ms'])

print('=' * 60)
print('RESUME LINE — use these exact numbers')
print('=' * 60)
print()
print(f"Model:        {chronicle['model_name']} (fp16, {chronicle['device']})")
print(f"Throughput:   {chronicle['tokens_per_sec']:.0f} tok/s Chronicle vs "
      f"{baseline['tokens_per_sec']:.0f} tok/s baseline ({gain_pct:.0f}% gain)")
print(f"Load test:    p50={best_level['p50_ms']:.0f}ms  p95={best_level['p95_ms']:.0f}ms  "
      f"p99={best_level['p99_ms']:.0f}ms  @ {best_level['concurrency']} concurrent")
print()
print('Suggested resume line:')
print()
batch_sizes = chronicle.get('batch_sizes', [8])
bs = batch_sizes[0] if batch_sizes else 8
print(f"  Benchmarked on {chronicle['model_name'].split('/')[-1]} (fp16, A10 GPU): "
      f"{gain_pct:.0f}% throughput gain over HuggingFace Transformers baseline at batch size {bs}")
print(f"  Load tested via async httpx harness: <{best_level['p95_ms']:.0f}ms p95 at "
      f"{best_level['concurrency']} concurrent requests; "
      f"p99 under {best_level['p99_ms']:.0f}ms with adaptive batch scheduling")

## Cleanup

In [ ]:
server_proc.terminate()
server_proc.wait()
print('Server stopped.')